In [1]:
import pandas as pd
import re

# Q1


In [2]:
medals = pd.read_csv('olympic_medals.csv')
hosts = pd.read_csv('olympic_hosts.csv')

In [3]:
medals.head()

,discipline_title,slug_game,event_title,event_gender,medal_type,participant_type,participant_title,athlete_url,athlete_full_name,country_name,country_code,country_3_letter_code
0,Curling,beijing-2022,Mixed Doubles,Mixed,GOLD,GameTeam,Italy,https://olympics.com/en/athletes/stefania-cons...,Stefania CONSTANTINI,Italy,IT,ITA
1,Curling,beijing-2022,Mixed Doubles,Mixed,GOLD,GameTeam,Italy,https://olympics.com/en/athletes/amos-mosaner,Amos MOSANER,Italy,IT,ITA
2,Curling,beijing-2022,Mixed Doubles,Mixed,SILVER,GameTeam,Norway,https://olympics.com/en/athletes/kristin-skaslien,Kristin SKASLIEN,Norway,NO,NOR
3,Curling,beijing-2022,Mixed Doubles,Mixed,SILVER,GameTeam,Norway,https://olympics.com/en/athletes/magnus-nedreg...,Magnus NEDREGOTTEN,Norway,NO,NOR
4,Curling,beijing-2022,Mixed Doubles,Mixed,BRONZE,GameTeam,Sweden,https://olympics.com/en/athletes/almida-de-val,Almida DE VAL,Sweden,SE,SWE


In [4]:
medals['year'] = medals['slug_game'].str.extract(r'(\d{4})').astype(int)

In [5]:
medals['year']

0        2022
1        2022
2        2022
3        2022
4        2022
         ... 
21692    1896
21693    1896
21694    1896
21695    1896
21696    1896
Name: year, Length: 21697, dtype: int64

In [6]:
hosts.head()

,game_slug,game_end_date,game_start_date,game_location,game_name,game_season,game_year
0,beijing-2022,2022-02-20T12:00:00Z,2022-02-04T15:00:00Z,China,Beijing 2022,Winter,2022
1,tokyo-2020,2021-08-08T14:00:00Z,2021-07-23T11:00:00Z,Japan,Tokyo 2020,Summer,2020
2,pyeongchang-2018,2018-02-25T08:00:00Z,2018-02-08T23:00:00Z,Republic of Korea,PyeongChang 2018,Winter,2018
3,rio-2016,2016-08-21T21:00:00Z,2016-08-05T12:00:00Z,Brazil,Rio 2016,Summer,2016
4,sochi-2014,2014-02-23T16:00:00Z,2014-02-07T04:00:00Z,Russian Federation,Sochi 2014,Winter,2014


In [8]:
hosts_slim = hosts[['game_slug', 'game_season', 'game_year']].copy()

In [9]:
hosts_slim.head()

,game_slug,game_season,game_year
0,beijing-2022,Winter,2022
1,tokyo-2020,Summer,2020
2,pyeongchang-2018,Winter,2018
3,rio-2016,Summer,2016
4,sochi-2014,Winter,2014


In [10]:
medals = medals.merge(hosts_slim, left_on = 'slug_game', right_on = 'game_slug', how='left' )

In [11]:
medals.head()

,discipline_title,slug_game,event_title,event_gender,medal_type,participant_type,participant_title,athlete_url,athlete_full_name,country_name,country_code,country_3_letter_code,year,game_slug,game_season,game_year
0,Curling,beijing-2022,Mixed Doubles,Mixed,GOLD,GameTeam,Italy,https://olympics.com/en/athletes/stefania-cons...,Stefania CONSTANTINI,Italy,IT,ITA,2022,beijing-2022,Winter,2022
1,Curling,beijing-2022,Mixed Doubles,Mixed,GOLD,GameTeam,Italy,https://olympics.com/en/athletes/amos-mosaner,Amos MOSANER,Italy,IT,ITA,2022,beijing-2022,Winter,2022
2,Curling,beijing-2022,Mixed Doubles,Mixed,SILVER,GameTeam,Norway,https://olympics.com/en/athletes/kristin-skaslien,Kristin SKASLIEN,Norway,NO,NOR,2022,beijing-2022,Winter,2022
3,Curling,beijing-2022,Mixed Doubles,Mixed,SILVER,GameTeam,Norway,https://olympics.com/en/athletes/magnus-nedreg...,Magnus NEDREGOTTEN,Norway,NO,NOR,2022,beijing-2022,Winter,2022
4,Curling,beijing-2022,Mixed Doubles,Mixed,BRONZE,GameTeam,Sweden,https://olympics.com/en/athletes/almida-de-val,Almida DE VAL,Sweden,SE,SWE,2022,beijing-2022,Winter,2022


In [12]:
medals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21697 entries, 0 to 21696
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   discipline_title       21697 non-null  object
 1   slug_game              21697 non-null  object
 2   event_title            21697 non-null  object
 3   event_gender           21697 non-null  object
 4   medal_type             21697 non-null  object
 5   participant_type       21697 non-null  object
 6   participant_title      6584 non-null   object
 7   athlete_url            17027 non-null  object
 8   athlete_full_name      18073 non-null  object
 9   country_name           21697 non-null  object
 10  country_code           20195 non-null  object
 11  country_3_letter_code  21697 non-null  object
 12  year                   21697 non-null  int64 
 13  game_slug              21697 non-null  object
 14  game_season            21697 non-null  object
 15  game_year          

In [13]:
hosts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   game_slug        53 non-null     object
 1   game_end_date    53 non-null     object
 2   game_start_date  53 non-null     object
 3   game_location    53 non-null     object
 4   game_name        53 non-null     object
 5   game_season      53 non-null     object
 6   game_year        53 non-null     int64 
dtypes: int64(1), object(6)
memory usage: 3.0+ KB


In [14]:
print('Seasons in data:', medals['game_season'].unique())
print('Total rows:', len(medals))

Seasons in data: ['Winter' 'Summer']
Total rows: 21697


In [15]:
#  Count medals per nation per edition per season
# Each row in olympic_medals.csv = one medal awarded grouping and counting rows gives medal count per group
medal_counts = medals.groupby([ 'country_3_letter_code',
    'country_name','year', 'game_season']).size().reset_index(name='medal_count')

In [16]:
medal_counts

,country_3_letter_code,country_name,year,game_season,medal_count
0,AFG,Afghanistan,2008,Summer,1
1,AFG,Afghanistan,2012,Summer,1
2,AHO,Netherlands Antilles,1988,Summer,1
3,ALG,Algeria,1984,Summer,2
4,ALG,Algeria,1992,Summer,2
...,...,...,...,...,...
1774,ZAM,Zambia,1984,Summer,1
1775,ZAM,Zambia,1996,Summer,1
1776,ZIM,Zimbabwe,1980,Summer,1
1777,ZIM,Zimbabwe,2004,Summer,3


In [17]:
# Compute total medals per edition per season
# Normalisation denominator: all medals awarded in that year + season this removes bias caused by the programme growing over time
# Summer grew from 43 events (1896) to 300+ events (2020)
# Without normalisation later nations appear stronger simply due to more medals
edition_totals = (medal_counts.groupby(['year','game_season'])['medal_count'].sum().reset_index(name='total_medals'))

In [18]:
edition_totals

,year,game_season,total_medals
0,1896,Summer,126
1,1900,Summer,292
2,1904,Summer,290
3,1908,Summer,343
4,1912,Summer,323
5,1920,Summer,457
6,1924,Summer,395
7,1924,Winter,52
8,1928,Summer,333
9,1928,Winter,44


In [19]:
# Merge totals and compute medal share 
# medal_share = (nation medals / edition total) * 100
medal_counts = medal_counts.merge(edition_totals, on=['year','game_season'])

medal_counts['medal_share'] = (medal_counts['medal_count'] / medal_counts['total_medals'] * 100).round(2)

In [20]:
medal_counts.head()

,country_3_letter_code,country_name,year,game_season,medal_count,total_medals,medal_share
0,AFG,Afghanistan,2008,Summer,1,1047,0.10
1,AFG,Afghanistan,2012,Summer,1,1044,0.10
2,AHO,Netherlands Antilles,1988,Summer,1,797,0.13
3,ALG,Algeria,1984,Summer,2,730,0.27
4,ALG,Algeria,1992,Summer,2,887,0.23


In [21]:
# Remove sparse nations - Nations with fewer than 5 medal-winning editions in a season
# are removed to avoid misleading single-point spikes in the line chart
editions_count = (medal_counts.groupby(['country_3_letter_code','game_season'])['year'].nunique().reset_index(name='edition_count'))

active = editions_count[editions_count['edition_count'] >= 5]

medal_counts = medal_counts.merge(
    active[['country_3_letter_code','game_season']],
    on=['country_3_letter_code','game_season'],
    how='inner'
)


In [22]:
medal_counts.to_csv('medals_clean.csv', index=False)

print(f'Exported {len(medal_counts)} rows')

print(f'Nations (Summer): {medal_counts[medal_counts.game_season=="Summer"].country_3_letter_code.nunique()}')
print(f'Nations (Winter): {medal_counts[medal_counts.game_season=="Winter"].country_3_letter_code.nunique()}')
print(medal_counts.head())


Exported 1609 rows
Nations (Summer): 84
Nations (Winter): 32
  country_3_letter_code country_name  year game_season  medal_count  \
0                   ALG      Algeria  1984      Summer            2   
1                   ALG      Algeria  1992      Summer            2   
2                   ALG      Algeria  1996      Summer            3   
3                   ALG      Algeria  2000      Summer            5   
4                   ALG      Algeria  2008      Summer            2   

   total_medals  medal_share  
0           730         0.27  
1           887         0.23  
2           917         0.33  
3          1023         0.49  
4          1047         0.19  


# Q2

In [23]:
hosts.head()

,game_slug,game_end_date,game_start_date,game_location,game_name,game_season,game_year
0,beijing-2022,2022-02-20T12:00:00Z,2022-02-04T15:00:00Z,China,Beijing 2022,Winter,2022
1,tokyo-2020,2021-08-08T14:00:00Z,2021-07-23T11:00:00Z,Japan,Tokyo 2020,Summer,2020
2,pyeongchang-2018,2018-02-25T08:00:00Z,2018-02-08T23:00:00Z,Republic of Korea,PyeongChang 2018,Winter,2018
3,rio-2016,2016-08-21T21:00:00Z,2016-08-05T12:00:00Z,Brazil,Rio 2016,Summer,2016
4,sochi-2014,2014-02-23T16:00:00Z,2014-02-07T04:00:00Z,Russian Federation,Sochi 2014,Winter,2014


In [24]:
host_years = hosts[hosts['game_season'].isin(['Summer','Winter'])][
    ['game_year','game_season','game_location']
].copy()
host_years = host_years.rename(columns={'game_year':'year'})

In [25]:
def get_host_code(year, season, medal_counts):
    edition = medal_counts[
        (medal_counts['year'] == year) &
        (medal_counts['game_season'] == season)
    ]
    if len(edition) == 0:
        return None
    return edition.loc[edition['medal_count'].idxmax(), 'country_3_letter_code']

In [27]:
# host_country_map = (
#     medals[['game_slug','country_name','country_3_letter_code','year','game_season','game_location']]
#     .drop_duplicates()
# )

In [28]:
host_entries = []
for _, row in host_years.iterrows():
    yr   = row['year']
    seas = row['game_season']
    loc  = str(row['game_location']).lower()
    # get all countries in that edition
    edition_countries = medal_counts[
        (medal_counts['year'] == yr) &
        (medal_counts['game_season'] == seas)
    ][['country_name','country_3_letter_code']].drop_duplicates()
    # try to match location to a country name
    match = None
    for _, c in edition_countries.iterrows():
        if any(part in loc for part in c['country_name'].lower().split()):
            match = c['country_3_letter_code']
            break
    host_entries.append({'year': yr, 'game_season': seas, 'host_code': match})

In [29]:
host_df = pd.DataFrame(host_entries)
# Drop rows where we couldn't match a host country
host_df = host_df.dropna(subset=['host_code'])

In [30]:
# For each host, label editions relative to hosting year 
results = []
for _, hrow in host_df.iterrows():
    code      = hrow['host_code']
    host_year = hrow['year']
    season    = hrow['game_season']

    # get all editions for this nation in this season
    nation = medal_counts[
        (medal_counts['country_3_letter_code'] == code) &
        (medal_counts['game_season'] == season)
    ].copy()

    # compute relative edition label
    nation['relative_edition'] = nation['year'] - host_year

    # keep only the 5 editions centred on hosting year
    nation = nation[nation['relative_edition'].between(-2, 2)].copy()
    nation['host_year']  = host_year
    nation['host_nation'] = code
    results.append(nation)

if results:
    q2_clean = pd.concat(results, ignore_index=True)
else:
    q2_clean = pd.DataFrame()

In [31]:
#   Paralympics host data 
para_summer = pd.read_csv('summer_paralympics.csv')
para_winter = pd.read_csv('winter_paralympics.csv')

In [32]:
# Compute medal totals and share for each nation per edition
for para_df, season_label in [(para_summer,'Summer'),(para_winter,'Winter')]:
    para_df['total_medals_nation'] = para_df['Gold'] + para_df['Silver'] + para_df['Bronze']
    ed_totals = para_df.groupby('Year')['total_medals_nation'].sum().reset_index(name='edition_total')
    para_df = para_df.merge(ed_totals, on='Year')
    para_df['medal_share'] = (para_df['total_medals_nation'] / para_df['edition_total'] * 100).round(2)
    para_df['game_season'] = season_label
    para_df.to_csv(f'q2_para_{season_label.lower()}_clean.csv', index=False)

In [33]:
# Export Q2 Olympic data 
q2_clean.to_csv('q2_host_clean.csv', index=False)
print(f'Q2 Olympic: {len(q2_clean)} rows')
print(q2_clean.head())

Q2 Olympic: 53 rows
  country_3_letter_code                country_name  year game_season  \
0                   CHN  People's Republic of China  2022      Winter   
1                   JPN                       Japan  2020      Summer   
2                   CHN  People's Republic of China  2018      Winter   
3                   BRA                      Brazil  2016      Summer   
4                   RUS          Russian Federation  2014      Winter   

   medal_count  total_medals  medal_share  relative_edition  host_year  \
0           16           355         4.51                 0       2022   
1           60          1188         5.05                 0       2020   
2           10           331         3.02                 0       2018   
3           23          1063         2.16                 0       2016   
4           33           314        10.51                 0       2014   

  host_nation  
0         CHN  
1         JPN  
2         CHN  
3         BRA  
4         RUS  


## Q3

In [34]:
# Step 3: Count women's event medals per nation per edition 
# event_gender column has values: 'Men', 'Women', 'Mixed'
# filter to Women events only to count women's medals
womens = medals[medals['event_gender'] == 'Women'].copy()

womens_counts = womens.groupby([
    'country_3_letter_code','year','game_season'
]).size().reset_index(name='womens_medal_count')

In [35]:
# Count total events per edition per season 
# Also compute how many women's events existed that edition
# to understand the share of women's events in the programme
womens_events_per_edition = (
    womens.groupby(['year','game_season'])['event_title']
    .nunique()
    .reset_index(name='womens_events_in_programme')
)
total_events_per_edition = (
    medals.groupby(['year','game_season'])['event_title']
    .nunique()
    .reset_index(name='total_events')
)

In [36]:
# Merge everything together
q3 = medal_counts.merge(
    womens_counts,
    on=['country_3_letter_code','year','game_season'],
    how='left'
)
q3['womens_medal_count'] = q3['womens_medal_count'].fillna(0)

# women's medal share = women's medals / total medals for that nation
q3['womens_medal_share'] = (
    q3['womens_medal_count'] / q3['medal_count'] * 100
).round(2)

# merge in programme-level context
q3 = q3.merge(womens_events_per_edition, on=['year','game_season'], how='left')
q3 = q3.merge(total_events_per_edition,  on=['year','game_season'], how='left')
# programme women's share = women's events / all events that edition
q3['programme_womens_share'] = (
    q3['womens_events_in_programme'] / q3['total_events'] * 100
).round(2)

In [37]:
# Prepare Paralympics gender data 
para_summer = pd.read_csv('summer_paralympics.csv')
para_winter = pd.read_csv('winter_paralympics.csv')

In [38]:
for para_df, label in [(para_summer,'Summer'),(para_winter,'Winter')]:
    # women's participation rate
    para_df['womens_participation_rate'] = (
        para_df['Women'] / para_df['P_Total'] * 100
    ).round(2)
    # total medals per nation
    para_df['total_medals_nation'] = para_df['Gold'] + para_df['Silver'] + para_df['Bronze']
    # edition total
    ed_tot = para_df.groupby('Year')['total_medals_nation'].sum().reset_index(name='edition_total')
    para_df = para_df.merge(ed_tot, on='Year')
    para_df['medal_share'] = (
        para_df['total_medals_nation'] / para_df['edition_total'] * 100
    ).round(2)
    para_df['game_season'] = label
    para_df.to_csv(f'q3_para_{label.lower()}_clean.csv', index=False)
    print(f'Para {label}: {len(para_df)} rows saved')

Para Summer: 1381 rows saved
Para Winter: 376 rows saved


In [39]:
# Remove sparse nations 
editions_count = (
    q3.groupby(['country_3_letter_code','game_season'])['year']
    .nunique()
    .reset_index(name='n_editions')
)
active = editions_count[editions_count['n_editions'] >= 5]
q3 = q3.merge(
    active[['country_3_letter_code','game_season']],
    on=['country_3_letter_code','game_season'],
    how='inner'
)

In [40]:
# Export 
q3.to_csv('q3_olympic_clean.csv', index=False)
print(f'Q3 Olympic: {len(q3)} rows')
print(q3.columns.tolist())
print(q3.head())

Q3 Olympic: 1609 rows
['country_3_letter_code', 'country_name', 'year', 'game_season', 'medal_count', 'total_medals', 'medal_share', 'womens_medal_count', 'womens_medal_share', 'womens_events_in_programme', 'total_events', 'programme_womens_share']
  country_3_letter_code country_name  year game_season  medal_count  \
0                   ALG      Algeria  1984      Summer            2   
1                   ALG      Algeria  1992      Summer            2   
2                   ALG      Algeria  1996      Summer            3   
3                   ALG      Algeria  2000      Summer            5   
4                   ALG      Algeria  2008      Summer            2   

   total_medals  medal_share  womens_medal_count  womens_medal_share  \
0           730         0.27                 0.0                 0.0   
1           887         0.23                 1.0                50.0   
2           917         0.33                 0.0                 0.0   
3          1023         0.49        